In [ ]:
import cv2
import matplotlib.pyplot as plt
import os
import glob # Import the glob library

from google.colab import drive
drive.mount('/content/drive')

# --- CONFIGURATION PARAMETERS ---
# Define the size of the patch to save (in pixels)
PATCH_SIZE = 64
BW, BH = PATCH_SIZE, PATCH_SIZE

# Define the paths
template_path = '/content/drive/MyDrive/PCB_DATASET/PCB_DATASET/PCB_USED/04.JPG'
folder_path = '/content/drive/MyDrive/PCB_DATASET/PCB_DATASET/images/Missing_hole/04_missing_hole'
out_dir = "/content/drive/MyDrive/PCB_DATASET/PCB_DATASET/Missing_hole_defects"

# Ensure the output directory exists
os.makedirs(out_dir, exist_ok=True)

# Global counter for saved patches - MUST be initialized outside the loop
global_patch_id = 0

# Define the area constraints for defect detection
MIN_AREA = 0
MAX_AREA = 1500


# Read template PCB 01 image
rgb_template_img = cv2.imread(template_path)
plt.figure(figsize=(10,6))
plt.imshow(rgb_template_img, cmap="gray")

# Calculate the resized dimensions (1/4th of original)
a = int(rgb_template_img.shape[0]/4)
b = int(rgb_template_img.shape[1]/4)

# Read template PCB 01 image as grayscale
template_img = cv2.imread(template_path, 0)
# display the grayscale template PCB image
plt.figure(figsize=(10,6))
plt.imshow(template_img, cmap="gray")

# Resize template image of PCB
template_img_resize = cv2.resize(template_img, (b,a))
plt.figure(figsize=(10,6))
plt.imshow(template_img_resize, cmap="gray")

# Gaussian blur
blur_template_img = cv2.GaussianBlur(template_img_resize, (3,3),0)
# display the blurred image
plt.figure(figsize=(10,6))
plt.imshow(blur_template_img, cmap="gray")

# Adaptive thresholding
template_adap_thresh = cv2.adaptiveThreshold(blur_template_img, 255,
                                         cv2. ADAPTIVE_THRESH_MEAN_C,
                                         cv2.THRESH_BINARY, 15, 5)
# display the thresholded image
plt.figure(figsize=(10,6))
plt.imshow(template_adap_thresh, cmap="gray")
# Note: Initial display code for template is removed here to keep the final script cleaner,
# but it can be re-added before the loop if needed for verification.


# Use glob to find all files ending with '.jpg' in the specified folder
image_files = glob.glob(os.path.join(folder_path, '*.jpg'))

print(f"Found {len(image_files)} images in the folder.")
print(f"Defect area filter range: {MIN_AREA} < Area < {MAX_AREA}")



# Iterate over each image file path found
for image_path in image_files:
    # 1. Read and prepare test image
    bgr_test_img = cv2.imread(image_path)
    filename = os.path.basename(image_path)

    # Check if the image was loaded successfully
    if bgr_test_img is None:
        print(f"Skipping file: {filename} (Not a readable image).")
        continue

    # Convert BGR to RGB for matplotlib display (if needed)
    rgb_test_img = cv2.cvtColor(bgr_test_img, cv2.COLOR_BGR2RGB) # Not used below, but good practice
     # 2. Display the original test PCB image
    plt.figure(figsize=(10,6))
    plt.imshow(rgb_test_img, cmap = "gray")
    plt.title(f"Original Test PCB Image: {filename}")

    # Read grayscale
    test_img = cv2.imread(image_path, 0)

    # Resize, Blur, and Threshold the test image
    test_img_resize = cv2.resize(test_img, (b, a))
    # 3. Display the grayscale resized test PCB image
    plt.figure(figsize=(10,6))
    plt.imshow(test_img_resize, cmap="gray")
    plt.title(f"Resized Grayscale PCB Image: {filename} ({b}x{a})")
    blur_test_img = cv2.GaussianBlur(test_img_resize, (3,3),0)

    test_adap_thresh = cv2.adaptiveThreshold(blur_test_img, 255,
                                         cv2. ADAPTIVE_THRESH_MEAN_C,
                                         cv2.THRESH_BINARY, 15, 5)

    plt.figure(figsize=(10,6))
    plt.imshow(test_adap_thresh, cmap="gray")

    # 2. Difference Imaging (Core Defect Detection)
    sub_img = cv2.subtract(test_adap_thresh, template_adap_thresh)
    plt.figure(figsize=(10,6))
    plt.imshow(sub_img)
    final_img = cv2.medianBlur(sub_img, 3) # Noise reduction
    plt.figure(figsize=(10,6))
    plt.imshow(final_img, cmap="gray")

    # 3. Find Contours
    # Using RETR_EXTERNAL focuses on the outline of defect blobs
    # Note: final_img is the mask image used for finding contours

    orig = test_img_resize
    mask_img = final_img
    _, thresh = cv2.threshold(mask_img, 127, 255, cv2.THRESH_BINARY)
    contours, _ = cv2.findContours(thresh, cv2.RETR_LIST,
    cv2.CHAIN_APPROX_SIMPLE) # CHANGE ***

    current_defects_count = 0
    orig = test_img_resize # The source image for cropping
    h_img, w_img = final_img.shape[:2]


    # 4. Filter Contours and Save Patches (MUST be inside the image loop)
    for cnt in contours:
        area = cv2.contourArea(cnt)

        # Apply the defect area filter to ensure quality and control quantity
        if MIN_AREA < area < MAX_AREA:
            current_defects_count += 1

            # Get bounding box coordinates
            x, y, w, h = cv2.boundingRect(cnt)

            # Center of defect
            cx = x + w // 2
            cy = y + h // 2

            # Fixed PATCh_SIZE x PATCh_SIZE box centered on defect
            x0 = cx - BW // 2
            y0 = cy - BH // 2
            x1 = x0 + BW
            y1 = y0 + BH

            # Clip coordinates to image bounds
            x0 = max(0, x0)
            y0 = max(0, y0)
            x1 = min(w_img, x1)
            y1 = min(h_img, y1)

            # Crop patch from ORIGINAL resized image
            patch = orig[y0:y1, x0:x1]

            # Ensure patch is the full desired size
            if patch.shape[0] != BH or patch.shape[1] != BW:
                continue

            # Save patch
            base_name = os.path.splitext(filename)[0]
            # Use global_patch_id for unique sequential naming across ALL files
            out_path = os.path.join(out_dir, f"defect_{base_name}_{global_patch_id:04d}.png")
            cv2.imwrite(out_path, patch)

            # Increment the global counter
            global_patch_id += 1

    print(f"Processed {filename}: Found {current_defects_count} defects (Total saved: {global_patch_id})")

print(f"\n--- Processing Complete ---")
print(f"Total patches saved to {out_dir}: {global_patch_id}")

In [ ]:
import cv2
import matplotlib.pyplot as plt
import os
import glob # Import the glob library

from google.colab import drive
drive.mount('/content/drive')

# --- CONFIGURATION PARAMETERS ---
# Define the size of the patch to save (in pixels)
PATCH_SIZE = 64
BW, BH = PATCH_SIZE, PATCH_SIZE

# Define the paths
template_path = '/content/drive/MyDrive/PCB_DATASET/PCB_DATASET/PCB_USED/12.JPG'
folder_path = '/content/drive/MyDrive/PCB_DATASET/PCB_DATASET/images/Spur/12_spur'
out_dir = "/content/drive/MyDrive/PCB_DATASET/PCB_DATASET/Spur_defects"

# Ensure the output directory exists
os.makedirs(out_dir, exist_ok=True)

# Global counter for saved patches - MUST be initialized outside the loop
global_patch_id = 0

# Define the area constraints for defect detection
MIN_AREA = 0
MAX_AREA = 300


# Read template PCB 01 image
rgb_template_img = cv2.imread(template_path)
plt.figure(figsize=(10,6))
plt.imshow(rgb_template_img, cmap="gray")

# Calculate the resized dimensions (1/4th of original)
a = int(rgb_template_img.shape[0]/4)
b = int(rgb_template_img.shape[1]/4)

# Read template PCB 01 image as grayscale
template_img = cv2.imread(template_path, 0)
# display the grayscale template PCB image
plt.figure(figsize=(10,6))
plt.imshow(template_img, cmap="gray")

# Resize template image of PCB
template_img_resize = cv2.resize(template_img, (b,a))
plt.figure(figsize=(10,6))
plt.imshow(template_img_resize, cmap="gray")

# Gaussian blur
blur_template_img = cv2.GaussianBlur(template_img_resize, (3,3),0)
# display the blurred image
plt.figure(figsize=(10,6))
plt.imshow(blur_template_img, cmap="gray")

# Adaptive thresholding
template_adap_thresh = cv2.adaptiveThreshold(blur_template_img, 255,
                                         cv2.ADAPTIVE_THRESH_MEAN_C,
                                         cv2.THRESH_BINARY, 25, 5)
# display the thresholded image
plt.figure(figsize=(10,6))
plt.imshow(template_adap_thresh, cmap="gray")
# Note: Initial display code for template is removed here to keep the final script cleaner,
# but it can be re-added before the loop if needed for verification.


# Use glob to find all files ending with '.jpg' in the specified folder
image_files = glob.glob(os.path.join(folder_path, '*.jpg'))

print(f"Found {len(image_files)} images in the folder.")
print(f"Defect area filter range: {MIN_AREA} < Area < {MAX_AREA}")



# Iterate over each image file path found
for image_path in image_files:
    # 1. Read and prepare test image
    bgr_test_img = cv2.imread(image_path)
    filename = os.path.basename(image_path)

    # Check if the image was loaded successfully
    if bgr_test_img is None:
        print(f"Skipping file: {filename} (Not a readable image).")
        continue

    # Convert BGR to RGB for matplotlib display (if needed)
    rgb_test_img = cv2.cvtColor(bgr_test_img, cv2.COLOR_BGR2RGB) # Not used below, but good practice
     # 2. Display the original test PCB image
    plt.figure(figsize=(10,6))
    plt.imshow(rgb_test_img, cmap = "gray")
    plt.title(f"Original Test PCB Image: {filename}")

    # Read grayscale
    test_img = cv2.imread(image_path, 0)

    # Resize, Blur, and Threshold the test image
    test_img_resize = cv2.resize(test_img, (b, a))
    # 3. Display the grayscale resized test PCB image
    plt.figure(figsize=(10,6))
    plt.imshow(test_img_resize, cmap="gray")
    plt.title(f"Resized Grayscale PCB Image: {filename} ({b}x{a})")
    blur_test_img = cv2.GaussianBlur(test_img_resize, (3,3),0)

    test_adap_thresh = cv2.adaptiveThreshold(blur_test_img, 255,
                                         cv2. ADAPTIVE_THRESH_MEAN_C,
                                         cv2.THRESH_BINARY, 25, 5)

    plt.figure(figsize=(10,6))
    plt.imshow(test_adap_thresh, cmap="gray")

    # 2. Difference Imaging (Core Defect Detection)
    sub_img = cv2.subtract(test_adap_thresh, template_adap_thresh)
    plt.figure(figsize=(10,6))
    plt.imshow(sub_img)
    final_img = cv2.medianBlur(sub_img, 3) # Noise reduction
    plt.figure(figsize=(10,6))
    plt.imshow(final_img, cmap="gray")

    # 3. Find Contours
    # Using RETR_EXTERNAL focuses on the outline of defect blobs
    # Note: final_img is the mask image used for finding contours

    orig = test_img_resize
    mask_img = final_img
    _, thresh = cv2.threshold(mask_img, 127, 255, cv2.THRESH_BINARY)
    contours, _ = cv2.findContours(thresh, cv2.RETR_LIST,
    cv2.CHAIN_APPROX_SIMPLE) # CHANGE ***

    current_defects_count = 0
    orig = test_img_resize # The source image for cropping
    h_img, w_img = final_img.shape[:2]


    # 4. Filter Contours and Save Patches (MUST be inside the image loop)
    for cnt in contours:
        area = cv2.contourArea(cnt)

        # Apply the defect area filter to ensure quality and control quantity
        if MIN_AREA < area < MAX_AREA:
            current_defects_count += 1

            # Get bounding box coordinates
            x, y, w, h = cv2.boundingRect(cnt)

            # Center of defect
            cx = x + w // 2
            cy = y + h // 2

            # Fixed PATCh_SIZE x PATCh_SIZE box centered on defect
            x0 = cx - BW // 2
            y0 = cy - BH // 2
            x1 = x0 + BW
            y1 = y0 + BH

            # Clip coordinates to image bounds
            x0 = max(0, x0)
            y0 = max(0, y0)
            x1 = min(w_img, x1)
            y1 = min(h_img, y1)

            # Crop patch from ORIGINAL resized image
            patch = orig[y0:y1, x0:x1]

            # Ensure patch is the full desired size
            if patch.shape[0] != BH or patch.shape[1] != BW:
                continue

            # Save patch
            base_name = os.path.splitext(filename)[0]
            # Use global_patch_id for unique sequential naming across ALL files
            out_path = os.path.join(out_dir, f"defect_{base_name}_{global_patch_id:04d}.png")
            cv2.imwrite(out_path, patch)

            # Increment the global counter
            global_patch_id += 1

    print(f"Processed {filename}: Found {current_defects_count} defects (Total saved: {global_patch_id})")

print(f"\n--- Processing Complete ---")
print(f"Total patches saved to {out_dir}: {global_patch_id}")